This workbook allows to list tables with row_count, example of rows from tables and list of expectations metrics (records passed and failed)

In [0]:
# configuration (choose catalog and provide pipeline id from its settings)
catalog= 'dev_catalog' #use prod_catalog or dev_catalog depending where your run this notebook
schemas = ['bronze','silver','golden']

pipeline_id='<your_pipeline_id_here>'

In [0]:
# list all tables with row counts
for schema in schemas:
    tables = [row.tableName for row in spark.sql(f"show tables in {catalog}.{schema}").collect()]
    for table in tables:
        row_count = spark.table(f'{catalog}.{schema}.{table}').count()
        print(f"table {catalog}.{schema}.{table} has {row_count} rows")

In [0]:
# show one row from all tables in each of the bronze, silver, golden schema
for schema in schemas:
    tables = [row.tableName for row in spark.sql(f"show tables in {catalog}.{schema}").collect()]
    for table in tables:
        print(f"{catalog}.{schema}.{table}")
        spark.sql(f"select * from {catalog}.{schema}.{table}").limit(1).display()

In [0]:
# show passed_records, failed_records per pipeline run, dataset name and the expectation name 
# data quality metrics are availble for event type = flow_progress and are stored in details:flow_progress:data_quality:expectations array
# expectations is an array of structs with name, dataset, passed_records, failed_records
# origin contains update_id (pipeline run id)

spark.sql(f"""
WITH parsed AS (
    SELECT
        origin.update_id as update_id,
        timestamp,
        parse_json(details):flow_progress:data_quality:expectations AS expectations
    FROM event_log('{pipeline_id}')
    WHERE event_type = 'flow_progress'
)
SELECT
    update_id,
    timestamp,
    e:name              AS expectation_name,
    e:dataset           AS dataset,
    e:passed_records    AS passed_records,
    e:failed_records    AS failed_records
FROM parsed
LATERAL VIEW explode(
    CAST(expectations AS ARRAY<VARIANT>)
) t AS e
""").display()